In [1]:
# Directories
import os
os.chdir(r"E:\academy\2. IIT PALAKKAD\1. academic\1. RESEARCH\PAPER\4.2 PAPER 3 FINAL\ANALYSIS")
print(os.getcwd())

E:\academy\2. IIT PALAKKAD\1. academic\1. RESEARCH\PAPER\4.2 PAPER 3 FINAL\ANALYSIS


In [2]:
#Array packages
import pandas as pd
import numpy as np
import xarray as xr
import dask
import netCDF4
#import h5netcdf

#computation
import scipy

#plots
import matplotlib.pyplot as plt
import rioxarray as rio
import geopandas as gpd
from shapely.geometry import mapping


# Directories
import os
import glob



C:\Users\sstar\anaconda3\envs\imed\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# 1_IMD DATA

## 1_IMD DATA Combining

In [5]:
#Dates
dt=pd.date_range(start='1950-01-01', end='2021-01-31',freq='M')

C:\Users\sstar\AppData\Local\Temp\ipykernel_12224\3293113032.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dt=pd.date_range(start='1950-01-01', end='2021-01-31',freq='M')


In [6]:
#1. LOADING IMD PRECIPITATION
year=np.arange(1950,2021,1)
files=sorted(glob.glob('DATA_r/3_IMD/RF25_ind*'+'rfp25.nc'))
imd1 = xr.open_mfdataset(files, combine='by_coords',engine='netcdf4', parallel=True,chunks={'time': 'auto'})
# Making name consistent
imd_new = imd1.rename({'LONGITUDE': 'lon', 'LATITUDE': 'lat', 'TIME': 'time'})
#imd = imd_new.transpose(*['time','lat','lon'])
imd_M=imd_new.resample(time='1M').sum()  #Monthly conversion


C:\Users\sstar\anaconda3\envs\imed\Lib\site-packages\xarray\core\groupby.py:532: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  index_grouper = pd.Grouper(


In [7]:
## Clipping to India shapefile
ind=gpd.read_file('Data_r/1_Shapefile/India_Boundary.shp')
imd_M.rio.set_spatial_dims(y_dim="lat",x_dim="lon", inplace=True)
imd_M.rio.write_crs("EPSG:4326", inplace=True)
clip_imd = imd_M.rio.clip(ind.geometry.apply(mapping), ind.crs, drop=True)
clip_imd.to_netcdf('Data_p/1_IMD data/IMD_ppt_0.25.nc')

## 2_IMD_DATA_SPI calculation
for spi calculation open jupyter lab/ active environment 'imed'

In [ ]:
from scipy.stats import gamma, norm

imd_M=xr.open_dataset('Data_p/1_IMD data/IMD_ppt_0.25.nc')
#imd_M = imd_M.where(imd_M.RAINFALL != 0, 0.01)
def spi_cal(data,yr):
    data1 = data.copy()

    if(np.sum(np.isnan(data1))>20):
        data1=data1
    else:
        for i in range(12): 
            sq=np.arange(i, i + 12*yr, 12)
            D=data1[sq]
            D[D == 0] = 0.01
            D1=D[~np.isnan(D)]
            shape, loc, scale = gamma.fit(D1,floc=0)
            POE = 1-gamma.cdf(D, shape, loc, scale)
            cdf_std = norm.cdf(norm.ppf(POE))
            spi = norm.ppf(1 - cdf_std)
            data1[sq]=spi
    return data1


scale=3
ds=imd_M.RAINFALL
ds_roll=ds.rolling(time=scale).mean()
SPI = xr.apply_ufunc(
    spi_cal,  # Function for wavelet transformation
    ds_roll,  # Input dataset
    input_core_dims=[['time']],  # Core dimensions of the input
    output_core_dims=[['time']],  # Core dimensions of the output
    vectorize=True,  # Treat each lat-lon pair as a separate input
    dask='parallelized',  # Use parallelized computation if using Dask arrays
    output_dtypes=[ds.dtype],  # Output data type
    output_sizes={'time': len(ds.time)},# Size of the output dimension
    kwargs={'yr': len(np.unique(ds['time.year']))}
)
final=xr.merge([imd_M,SPI.rename('SPI')])
final.to_netcdf('Data_p/1_IMD data/IMD_SPI_0.25.nc')

#spi=spi.where(~spi.time.isin(spi.time[range(scale-1)]),np.nan)

# 2_Dynamical Model data

Also you can direct download

url = 'http://iridl.ldeo.columbia.edu/SOURCES/.NOAA/.NCEP/.CPC/.CAMS_OPI/.v0208/.mean/.prcp/dods'
ds = xr.open_dataset(url,decode_times=False)
ds

### 2.1_Merging model outputs

In [3]:
dts=pd.date_range(start='1982-02-01', end='2011-01-31',freq='M')
nmme = {}
mdl=['CFSv2', 'CMC1-CanCM3', 'CMC2-CanCM4','GFDL-CM2p1-aer04','NASA-GMAO']
for m in mdl:

    ds= xr.open_dataset(f'DATA_r/2_Dynamical/{m}.nc',decode_times=False)
    ds = ds.rename({'X': 'lon', 'Y': 'lat', 'S': 'time'})
    ds=ds.mean(dim='L')
    ds['time'] = dts
    ds=ds.rename_vars({'prec':m})
    nmme[f'{m}'] = ds

#Regridding precipitation data
imd_M=xr.open_dataset('Data_p/1_IMD data/IMD_ppt_0.25.nc')
imd_dyn=imd_M.interp(lat=nmme[mdl[0]].lat,lon=nmme[mdl[0]].lon)
imd_dyn1=imd_dyn.sel(time=nmme[mdl[0]].time)



merged_nmme=xr.merge([nmme[mdl[0]], nmme[mdl[1]],nmme[mdl[2]],nmme[mdl[3]],nmme[mdl[4]],imd_dyn1['RAINFALL']])
merged_nmme.to_netcdf('Data_p/2_Dynamical/NMME.nc')


C:\Users\sstar\AppData\Local\Temp\ipykernel_20304\3117768536.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dts=pd.date_range(start='1982-02-01', end='2011-01-31',freq='M')


In [21]:
from scipy.stats import gamma, norm

dyn_M=xr.open_dataset('Data_p/2_Dynamical/NMME.nc')


def spi_cal(data,yr):
    data1 = data.copy()

    if(np.sum(np.isnan(data1))>20):
        data1=data1
    else:
        for i in range(12): 
            sq=np.arange(i, i + 12*yr, 12)
            D=data1[sq]
            D[D <= 0] = 0.01
            D1=D[~np.isnan(D)]
            D1=D1[~np.isinf(D1)]
            shape, loc, scale = gamma.fit(D1,floc=0)
            POE = 1-gamma.cdf(D, shape, loc, scale)
            cdf_std = norm.cdf(norm.ppf(POE))
            spi = norm.ppf(1 - cdf_std)
            data1[sq]=spi
    return data1


scale=3
ds=dyn_M.isel(time=dyn_M['time.year'].isin(range(1982,2011)))
ds_roll=ds.rolling(time=scale).mean()
SPI = xr.apply_ufunc(
    spi_cal,  # Function for wavelet transformation
    ds_roll,  # Input dataset
    input_core_dims=[['time']],  # Core dimensions of the input
    output_core_dims=[['time']],  # Core dimensions of the output
    vectorize=True,  # Treat each lat-lon pair as a separate input
    dask='parallelized',  # Use parallelized computation if using Dask arrays
    output_dtypes=[ds.RAINFALL.dtype],  # Output data type
    output_sizes={'time': len(ds.time)},# Size of the output dimension
    kwargs={'yr': len(np.unique(ds['time.year']))-1}
)

C:\Users\sstar\AppData\Local\Temp\ipykernel_20304\1314359349.py:29: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  SPI = xr.apply_ufunc(


In [ ]:
final=xr.merge([imd_M,SPI.rename('SPI')])
final.to_netcdf('Data_p/1_IMD data/IMD_SPI_0.25.nc')

#spi=spi.where(~spi.time.isin(spi.time[range(scale-1)]),np.nan)

ds=merged_nmme
dataset=[]
var=np.array(merged_nmme.data_vars)
for vr in range(len(var)):
    ds1=ds.rolling(time=scale).mean()
    spi = ds1[var[vr]].groupby('time.month').apply(calculate_spi)
    spi=spi.where(~spi.time.isin(spi.time[range(scale-1)]),np.nan)
    dataset.append(spi)


#to dataset
data_dict = {f'{dataset[i].name}': dataset[i] for i in range(len(dataset))}
fin = xr.Dataset(data_dict)

fin.to_netcdf('Data_p/1_IMD data/NMME_SPI.nc')